# 1. TMDB Movie Data Analysis  
## Fetch Movie Data from TMDb API

This fetches movie data from the TMDb API using Given movie IDs.
The data is stored in a Pandas DataFrame for further cleaning and analysis.


In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



### Movie IDs

The following list contains TMDb movie IDs that will be used to fetch movie details.
Invalid IDs (e.g., 0) will be skipped during the API request process.


In [2]:
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397,
    420818, 24428, 168259, 99861, 284054, 12445,
    181808, 330457, 351286, 109445, 321612, 260513
]


### TMDb API Key



In [3]:
API_KEY = "1a66cf9f283cad8d30e13096de0af285"


### API Base URL

This is the base endpoint for fetching movie details from TMDb.


In [4]:
BASE_URL = "https://api.themoviedb.org/3/movie/"

### Fetching Movie Data

Each movie ID is queried from the TMDb API.
Valid responses are stored as JSON objects in a list.


In [5]:
movies_data = []

for movie_id in movie_ids:

    # Skip invalid movie ID
    if movie_id == 0:
        continue

    url = f"{BASE_URL}{movie_id}"
    params = {"api_key": API_KEY}

    response = requests.get(url, params=params)

    if response.status_code == 200:
        movies_data.append(response.json())
    else:
        print(f"Failed to fetch movie ID {movie_id}")


### Pandas DataFrame

The collected movie data is converted into a Pandas DataFrame.
Each row represents one movie.


In [6]:
df_movies = pd.DataFrame(movies_data)


### Initial Data Inspection

We inspect the first few rows and dataset structure to understand the available columns.


In [7]:
df_movies.head()


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,2019-04-24,2799439100,181,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.238,27116
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,2009-12-16,2923706026,162,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.600,33224
2,False,/8BTsTfln4jlQrLXUBquXJ0ASQy9.jpg,"{'id': 10, 'name': 'Star Wars Collection', 'po...",245000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",http://www.starwars.com/films/star-wars-episod...,140607,tt2488496,[US],en,...,2015-12-15,2068223624,136,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Every generation has a story.,Star Wars: The Force Awakens,False,7.254,20190
3,False,/mDfJG3LC3Dqb67AZ52x3Z0jU0uB.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",300000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...",https://www.marvel.com/movies/avengers-infinit...,299536,tt4154756,[US],en,...,2018-04-25,2052415039,149,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Destiny arrives all the same.,Avengers: Infinity War,False,8.235,31330
4,False,/xnHVX37XZEp33hhCbYlQFq7ux1J.jpg,None,200000000,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",https://www.paramountmovies.com/movies/titanic,597,tt0120338,[US],en,...,1997-12-18,2264162353,194,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Nothing on earth could come between them.,Titanic,False,7.902,26648


In [8]:
df_movies.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  18 non-null     bool   
 1   backdrop_path          18 non-null     object 
 2   belongs_to_collection  16 non-null     object 
 3   budget                 18 non-null     int64  
 4   genres                 18 non-null     object 
 5   homepage               18 non-null     object 
 6   id                     18 non-null     int64  
 7   imdb_id                18 non-null     object 
 8   origin_country         18 non-null     object 
 9   original_language      18 non-null     object 
 10  original_title         18 non-null     object 
 11  overview               18 non-null     object 
 12  popularity             18 non-null     float64
 13  poster_path            18 non-null     object 
 14  production_companies   18 non-null     object 
 15  producti

# 2. Data Cleaning and Preprocessing

This section cleans and transforms the raw TMDb movie dataset obtained in Step 1.
The goal is to prepare a structured and reliable dataset for analysis.


## Drop Irrelevant Columns

The following columns are not required for analysis and are removed:
- adult
- imdb_id
- original_title
- video
- homepage


In [9]:
cols_to_drop = ['adult', 'imdb_id', 'original_title', 'video', 'homepage']
df_movies.drop(columns=cols_to_drop, inplace=True, errors='ignore')


## Extract JSON-like Columns

Several columns contain nested JSON objects or lists.
These columns are transformed into readable string formats.


In [10]:
def extract_names(json_list):
    if isinstance(json_list, list):
        return "|".join([item['name'] for item in json_list if 'name' in item])
    return np.nan


In [11]:
# Collection name
df_movies['belongs_to_collection'] = df_movies['belongs_to_collection'].apply(
    lambda x: x['name'] if isinstance(x, dict) else np.nan
)

# Genres
df_movies['genres'] = df_movies['genres'].apply(extract_names)

# Spoken languages
df_movies['spoken_languages'] = df_movies['spoken_languages'].apply(extract_names)

# Production countries
df_movies['production_countries'] = df_movies['production_countries'].apply(extract_names)

# Production companies
df_movies['production_companies'] = df_movies['production_companies'].apply(extract_names)


## Inspect Extracted Columns

We inspect value counts to identify anomalies or unexpected values.


In [12]:
df_movies['genres'].value_counts().head()


,count
genres,
Adventure|Action|Science Fiction,3
Action|Adventure|Science Fiction|Thriller,2
Action|Adventure|Science Fiction,2
Action|Adventure|Fantasy|Science Fiction,1
Drama|Romance,1


## Convert Data Types

I Convert columns to appropriate data types for analysis.


In [13]:
numeric_cols = [
    'budget', 'id', 'popularity', 'revenue',
    'runtime', 'vote_count', 'vote_average'
]

for col in numeric_cols:
    df_movies[col] = pd.to_numeric(df_movies[col], errors='coerce')

df_movies['release_date'] = pd.to_datetime(df_movies['release_date'], errors='coerce')


## Handle Missing and Unrealistic Values

- Replace 0 values in budget, revenue, and runtime with NaN
- Convert budget and revenue to million USD
- Replace placeholder text with NaN


In [14]:
# Replace unrealistic zeros
df_movies[['budget', 'revenue', 'runtime']] = (
    df_movies[['budget', 'revenue', 'runtime']].replace(0, np.nan)
)

# Convert to million USD
df_movies['budget_musd'] = df_movies['budget'] / 1_000_000
df_movies['revenue_musd'] = df_movies['revenue'] / 1_000_000

# Replace placeholder text
df_movies['overview'] = df_movies['overview'].replace('No Data', np.nan)
df_movies['tagline'] = df_movies['tagline'].replace('No Data', np.nan)


## Remove Invalid Rows

- Remove duplicates
- Drop rows with missing movie ID or title
- Keep rows with sufficient non-missing data


In [15]:
# Remove duplicate movies based on unique movie ID
df_movies.drop_duplicates(subset=['id'], inplace=True)

# Drop rows with missing ID or title
df_movies.dropna(subset=['id', 'title'], inplace=True)

# Keep rows with at least 10 non-NaN values
df_movies = df_movies[df_movies.notna().sum(axis=1) >= 10]


## Filter Released Movies

Only movies with status "Released" are included in the final dataset.


In [16]:
df_movies = df_movies[df_movies['status'] == 'Released']
df_movies.drop(columns='status', inplace=True)


## Reorder and Finalize Dataset

Columns are reordered according to the project specifications.


In [18]:
final_columns = [
    'id', 'title', 'tagline', 'release_date', 'genres',
    'belongs_to_collection', 'original_language',
    'budget_musd', 'revenue_musd',
    'production_companies', 'production_countries',
    'vote_count', 'vote_average', 'popularity', 'runtime',
    'overview', 'spoken_languages', 'poster_path'
]

df_movies = df_movies[final_columns]
df_movies.reset_index(drop=True, inplace=True)


## Fetch Cast and Director Information (Credits API)

This section enriches the dataset with cast and director information
required for advanced search queries and director analysis.


In [20]:
def fetch_credits(movie_id, api_key):
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    params = {"api_key": api_key}
    r = requests.get(url, params=params)

    if r.status_code != 200:
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    data = r.json()

    cast = data.get('cast', [])
    crew = data.get('crew', [])

    cast_names = "|".join([c['name'] for c in cast])
    director = next((c['name'] for c in crew if c['job'] == 'Director'), np.nan)

    return pd.Series([
        cast_names,
        len(cast),
        director,
        len(crew)
    ])


In [21]:
df_movies[['cast', 'cast_size', 'director', 'crew_size']] = (
    df_movies['id']
    .apply(lambda x: fetch_credits(x, API_KEY))
)


# 3. KPI Implementation & Analysis

This step computes key performance indicators (KPIs) to identify
the best and worst performing movies based on financial, popularity,
and rating metrics.


In [22]:
# Profit and ROI
df_movies['profit_musd'] = df_movies['revenue_musd'] - df_movies['budget_musd']
df_movies['roi'] = df_movies['revenue_musd'] / df_movies['budget_musd']


## Helper Function for Ranking Movies

This function ranks movies based on a metric and returns the top or bottom results.


In [23]:
def rank_movies(df, metric, n=5, ascending=False, min_budget=None, min_votes=None):
    temp = df.copy()

    if min_budget is not None:
        temp = temp[temp['budget_musd'] >= min_budget]

    if min_votes is not None:
        temp = temp[temp['vote_count'] >= min_votes]

    return temp.sort_values(by=metric, ascending=ascending).head(n)[
        ['title', metric]
    ]


## Best and Worst Performing Movies


In [24]:
# Highest Revenue
rank_movies(df_movies, 'revenue_musd')

# Highest Budget
rank_movies(df_movies, 'budget_musd')

# Highest Profit
rank_movies(df_movies, 'profit_musd')

# Lowest Profit
rank_movies(df_movies, 'profit_musd', ascending=True)


,title,profit_musd
12,Star Wars: The Last Jedi,1032.698830
17,Incredibles 2,1043.225667
16,Beauty and the Beast,1106.115964
15,Frozen,1124.219009
14,Jurassic World: Fallen Kingdom,1140.469037


## Return on Investment (ROI)
Only movies with a budget ≥ 10 million USD are considered.


In [25]:
# Highest ROI
rank_movies(df_movies, 'roi', min_budget=10)

# Lowest ROI
rank_movies(df_movies, 'roi', ascending=True, min_budget=10)


,title,roi
12,Star Wars: The Last Jedi,4.442329
9,Avengers: Age of Ultron,5.980441
17,Incredibles 2,6.216128
6,The Lion King,6.392388
10,Black Panther,6.749630


## Voting and Popularity Metrics


In [27]:
# Most Voted Movies
rank_movies(df_movies, 'vote_count')

# Highest Rated Movies (≥ 10 votes)
rank_movies(df_movies, 'vote_average', min_votes=10)

# Lowest Rated Movies (≥ 10 votes)
rank_movies(df_movies, 'vote_average', ascending=True, min_votes=10)

# Most Popular Movies
rank_movies(df_movies, 'popularity')


,title,popularity
1,Avatar,70.1711
7,The Avengers,65.5674
4,Titanic,35.3101
3,Avengers: Infinity War,28.8523
0,Avengers: Endgame,18.6659


## Advanced Movie Search Queries


###
Search 1

Best-rated Sci-Fi Action movies starring Bruce Willis

In [28]:
search_1 = df_movies[
    df_movies['genres'].str.contains('Science Fiction', na=False) &
    df_movies['genres'].str.contains('Action', na=False) &
    df_movies['cast'].str.contains('Bruce Willis', na=False)
].sort_values('vote_average', ascending=False)

search_1[['title', 'vote_average']]


,title,vote_average


### Search 2

Movies starring Uma Thurman, directed by Quentin Tarantino

In [29]:
search_2 = df_movies[
    df_movies['cast'].str.contains('Uma Thurman', na=False) &
    (df_movies['director'] == 'Quentin Tarantino')
].sort_values('runtime')

search_2[['title', 'runtime']]


,title,runtime


## Franchise vs Standalone Movie Performance


In [30]:
df_movies['is_franchise'] = df_movies['belongs_to_collection'].notna()

df_movies.groupby('is_franchise').agg(
    mean_revenue=('revenue_musd', 'mean'),
    median_roi=('roi', 'median'),
    mean_budget=('budget_musd', 'mean'),
    mean_popularity=('popularity', 'mean'),
    mean_rating=('vote_average', 'mean')
)


,mean_revenue,median_roi,mean_budget,mean_popularity,mean_rating
is_franchise,,,,,
False,1765.139159,9.617018,180.0,26.295950,7.4355
True,1682.668411,7.786117,218.0,20.526512,7.3855


## Most Successful Movie Franchises


In [31]:
franchise_stats = (
    df_movies.dropna(subset=['belongs_to_collection'])
    .groupby('belongs_to_collection')
    .agg(
        movie_count=('title', 'count'),
        total_budget=('budget_musd', 'sum'),
        mean_budget=('budget_musd', 'mean'),
        total_revenue=('revenue_musd', 'sum'),
        mean_revenue=('revenue_musd', 'mean'),
        mean_rating=('vote_average', 'mean')
    )
    .sort_values('total_revenue', ascending=False)
)

franchise_stats.head()


,movie_count,total_budget,mean_budget,total_revenue,mean_revenue,mean_rating
belongs_to_collection,,,,,,
The Avengers Collection,4,1111.0,277.75,7776.073348,1944.018337,7.91125
Star Wars Collection,2,545.0,272.50,3400.922454,1700.461227,7.00800
Jurassic Park Collection,2,320.0,160.00,2982.006481,1491.003241,6.60000
Avatar Collection,1,237.0,237.00,2923.706026,2923.706026,7.60000
Frozen Collection,2,300.0,150.00,2727.902485,1363.951242,7.24300


## Most Successful Directors


In [32]:
director_stats = (
    df_movies.dropna(subset=['director'])
    .groupby('director')
    .agg(
        movie_count=('title', 'count'),
        total_revenue=('revenue_musd', 'sum'),
        mean_rating=('vote_average', 'mean')
    )
    .sort_values('total_revenue', ascending=False)
)

director_stats.head()


,movie_count,total_revenue,mean_rating
director,,,
James Cameron,2,5187.868379,7.751
Joss Whedon,2,2924.219209,7.586
Anthony Russo,1,2799.439100,8.238
Jennifer Lee,2,2727.902485,7.243
J.J. Abrams,1,2068.223624,7.254
